# Wonderkid Scouting Report 2024-25
## Predicting Elite Status by 2027-28

This notebook applies our trained prediction model to current wonderkids (age 16-19) to identify who is most likely to become an elite player within the next 3 years.

### Methodology
- **Training Data:** 97 players from 2021-22 and 2022-23 seasons → 2024-25 outcomes
- **Model:** Random Forest with heavy regularization (CV AUC: 0.81)
- **Elite Definition:** Top 15% in Goals + Assists per 90 minutes
- **Prediction Window:** 2024-25 skills → 2027-28 projected outcomes

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import io
from difflib import SequenceMatcher
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded!")

## 1. Load and Prepare Data

In [ ]:
# Load all datasets
data_2122 = pd.read_csv('../data/raw/players_data-2021_2022.csv', sep=';', encoding='latin-1')
data_2223 = pd.read_csv('../data/raw/2022-2023 Football Player Stats.csv', sep=';', encoding='latin-1')
data_2425 = pd.read_csv('../data/raw/players_data-2024_2025.csv')

print(f"2021-22: {len(data_2122)} players")
print(f"2022-23: {len(data_2223)} players")
print(f"2024-25: {len(data_2425)} players")

# Column mapping
column_mapping = {
    'Goals': 'Gls', 'Assists': 'Ast', 'CarProg': 'PrgC', 'PasProg': 'PrgP',
    'RecProg': 'PrgR', 'PasTotCmp%': 'Cmp%', 'CarMis': 'Mis', 'CarDis': 'Dis',
    'PasTotPrgDist': 'PrgDist', 'ScaPassLive': 'SCA', 'GcaPassLive': 'GCA'
}

data_2122 = data_2122.rename(columns=column_mapping)
data_2223 = data_2223.rename(columns=column_mapping)

print("\nData loaded and columns standardized!")

## 2. Train Prediction Model

In [ ]:
# Features and helper functions
SKILL_FEATURES = ['Age', 'Cmp%', 'PrgDist', 'PrgP', 'PrgC', 'Mis', 'Dis',
                  'Tkl', 'Int', 'Clr', 'Blocks', 'Touches', 'Rec', 'SCA', 'GCA']

def find_best_match(player_name, candidates, threshold=0.85):
    best_match, best_score = None, 0
    for candidate in candidates:
        score = SequenceMatcher(None, str(player_name).lower(), str(candidate).lower()).ratio()
        if score > best_score and score >= threshold:
            best_score, best_match = score, candidate
    return best_match

# Prepare outcome data (2024-25 elite players)
outcome = data_2425[data_2425['Pos'].str.contains('FW|MF', na=False)].copy()
outcome = outcome[outcome['Min'] >= 450]
outcome = outcome.drop_duplicates(subset='Player')
outcome['GA90'] = (outcome['Gls'] + outcome['Ast']) / (outcome['Min'] / 90)
elite_threshold = outcome['GA90'].quantile(0.85)
outcome['is_elite'] = (outcome['GA90'] >= elite_threshold).astype(int)
outcome_players = outcome['Player'].unique()

print(f"Elite threshold: {elite_threshold:.2f} G+A/90")
print(f"Elite players in 2024-25: {outcome['is_elite'].sum()}")

In [ ]:
# Build training cohorts
def build_cohort(df, cohort_name, min_age=16, max_age=21):
    cohort = df[df['Pos'].str.contains('FW|MF', na=False)].copy()
    cohort = cohort[(cohort['Age'] >= min_age) & (cohort['Age'] <= max_age)]
    cohort = cohort[cohort['Min'] >= 450]
    cohort = cohort.drop_duplicates(subset='Player')
    
    tracked = []
    for _, row in cohort.iterrows():
        match = find_best_match(row['Player'], outcome_players)
        if match:
            outcome_row = outcome[outcome['Player'] == match].iloc[0]
            player_data = {'player': row['Player'], 'cohort': cohort_name, 'is_elite': outcome_row['is_elite']}
            for feat in SKILL_FEATURES:
                player_data[feat] = row[feat] if feat in row.index else np.nan
            tracked.append(player_data)
    return pd.DataFrame(tracked)

cohort_2122 = build_cohort(data_2122, '2021-22')
cohort_2223 = build_cohort(data_2223, '2022-23')

train_data = pd.concat([cohort_2122, cohort_2223], ignore_index=True)
train_data = train_data.sort_values('cohort', ascending=False).drop_duplicates(subset='player')

print(f"Training data: {len(train_data)} players")
print(f"Elite in training: {train_data['is_elite'].sum()} ({train_data['is_elite'].mean():.1%})")

In [ ]:
# Train model
X_train = train_data[SKILL_FEATURES].fillna(train_data[SKILL_FEATURES].median())
y_train = train_data['is_elite']

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

rf = RandomForestClassifier(
    n_estimators=50, max_depth=3, min_samples_split=12,
    min_samples_leaf=4, max_features=0.6, random_state=42,
    class_weight='balanced'
)
rf.fit(X_train_scaled, y_train)
print("✓ Model trained!")

## 3. Scout 2024-25 Wonderkids

In [ ]:
# Filter for young wonderkids
wonderkids = data_2425[data_2425['Pos'].str.contains('FW|MF', na=False)].copy()
wonderkids = wonderkids[(wonderkids['Age'] >= 16) & (wonderkids['Age'] <= 19)]
wonderkids = wonderkids[wonderkids['Min'] >= 300]  # At least some playing time
wonderkids = wonderkids.drop_duplicates(subset='Player')

print(f"Wonderkids found: {len(wonderkids)}")
print(f"Age distribution:")
print(wonderkids['Age'].value_counts().sort_index())

In [ ]:
# Predict elite probability
X_scout = wonderkids[SKILL_FEATURES].copy()
for col in X_scout.columns:
    X_scout[col] = pd.to_numeric(X_scout[col], errors='coerce')
    X_scout[col] = X_scout[col].fillna(X_scout[col].median())

X_scout_scaled = scaler.transform(X_scout)
wonderkids['elite_prob'] = rf.predict_proba(X_scout_scaled)[:, 1]
wonderkids['current_GA90'] = (wonderkids['Gls'] + wonderkids['Ast']) / (wonderkids['Min'] / 90)
wonderkids = wonderkids.sort_values('elite_prob', ascending=False)

print("✓ Predictions complete!")

## 4. Top 20 Prospects

In [ ]:
# Display top 20
print("=" * 80)
print("TOP 20 PROSPECTS - Predicted to be Elite by 2027-28")
print("=" * 80)
print()

top20 = wonderkids.head(20)
for i, (_, row) in enumerate(top20.iterrows(), 1):
    club = str(row['Squad'])[:18] if pd.notna(row['Squad']) else 'Unknown'
    confidence = "🟢" if row['elite_prob'] >= 0.60 else "🟡" if row['elite_prob'] >= 0.55 else "🟠"
    print(f"{i:2}. {confidence} {row['Player']:<22} Age {int(row['Age'])}  {club:<18}  {row['elite_prob']:.0%}  (G+A/90: {row['current_GA90']:.2f})")

## 5. Analysis by League

In [ ]:
# Top prospect per league
print("=" * 60)
print("TOP PROSPECT BY LEAGUE")
print("=" * 60)
print()

leagues = {
    'Premier League': 'eng Premier League',
    'La Liga': 'es La Liga', 
    'Bundesliga': 'de Bundesliga',
    'Serie A': 'it Serie A',
    'Ligue 1': 'fr Ligue 1'
}

for name, pattern in leagues.items():
    league_data = wonderkids[wonderkids['Comp'].str.contains(pattern, na=False)]
    if len(league_data) > 0:
        top = league_data.iloc[0]
        print(f"{name}:")
        print(f"   {top['Player']} ({top['Squad']}, age {int(top['Age'])})")
        print(f"   Elite probability: {top['elite_prob']:.0%}")
        print()

## 6. Probability Distribution

In [ ]:
# Visualize probability distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(wonderkids['elite_prob'], bins=20, color='steelblue', edgecolor='white')
axes[0].axvline(x=0.60, color='green', linestyle='--', label='High confidence (60%)')
axes[0].axvline(x=0.50, color='orange', linestyle='--', label='Medium confidence (50%)')
axes[0].set_xlabel('Elite Probability')
axes[0].set_ylabel('Number of Players')
axes[0].set_title('Distribution of Elite Probabilities')
axes[0].legend()

# Top 15 bar chart
top15 = wonderkids.head(15)
colors = ['green' if p >= 0.60 else 'gold' if p >= 0.55 else 'orange' for p in top15['elite_prob']]
axes[1].barh(range(len(top15)), top15['elite_prob'], color=colors)
axes[1].set_yticks(range(len(top15)))
axes[1].set_yticklabels(top15['Player'])
axes[1].set_xlabel('Elite Probability')
axes[1].set_title('Top 15 Prospects by Elite Probability')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## 7. Detailed Player Profiles

In [ ]:
# Top 5 detailed profiles
print("=" * 80)
print("DETAILED PROSPECT PROFILES")
print("=" * 80)

for i, (_, row) in enumerate(wonderkids.head(5).iterrows(), 1):
    print(f"\n{'─' * 60}")
    print(f"#{i} {row['Player'].upper()}")
    print(f"{'─' * 60}")
    print(f"Age: {int(row['Age'])} | Position: {row['Pos']}")
    print(f"Club: {row['Squad']} | League: {row['Comp']}")
    print(f"\nPerformance (2024-25):")
    print(f"   Minutes: {int(row['Min'])} | Goals: {int(row['Gls'])} | Assists: {int(row['Ast'])}")
    print(f"   Current G+A/90: {row['current_GA90']:.2f}")
    print(f"\n🎯 ELITE PROBABILITY: {row['elite_prob']:.1%}")
    
    # Skills comparison
    print(f"\nKey Skills:")
    for feat in ['Tkl', 'PrgC', 'SCA', 'GCA', 'Touches']:
        if feat in row.index and pd.notna(row[feat]):
            avg = wonderkids[feat].median()
            diff = ((row[feat] - avg) / avg) * 100 if avg > 0 else 0
            arrow = "↑" if diff > 20 else "↓" if diff < -20 else "→"
            print(f"   {feat}: {int(row[feat])} {arrow} ({diff:+.0f}% vs avg)")

## 8. Save Scouting Report

In [ ]:
# Save report
report_cols = ['Player', 'Age', 'Pos', 'Squad', 'Comp', 'Min', 'Gls', 'Ast', 
               'current_GA90', 'elite_prob']
report = wonderkids[report_cols].copy()
report.columns = ['Player', 'Age', 'Position', 'Club', 'League', 'Minutes', 
                  'Goals', 'Assists', 'Current_GA90', 'Elite_Probability']
report = report.sort_values('Elite_Probability', ascending=False)

report.to_csv('../data/processed/scouting_report_2024.csv', index=False)
print("✓ Scouting report saved to data/processed/scouting_report_2024.csv")
print(f"\nTotal prospects: {len(report)}")
print(f"High confidence (60%+): {(report['Elite_Probability'] >= 0.60).sum()}")
print(f"Medium confidence (50-59%): {((report['Elite_Probability'] >= 0.50) & (report['Elite_Probability'] < 0.60)).sum()}")

## Summary

### Key Findings

**#1 Prospect: Endrick (Real Madrid)**
- 18 years old, Brazilian striker
- 70% probability of becoming elite by 2027-28
- Already playing for the biggest club in the world

**Other Notable Prospects:**
- **Ethan Nwaneri** (Arsenal, 17) - Youngest on the list, already contributing (0.60 G+A/90)
- **Ibrahim Mbaye** (PSG, 16) - Incredibly young, making first-team appearances
- **Désiré Doué** (PSG, 19) - Already performing at elite level (0.62 G+A/90)
- **Nicolás Paz** (Como, 19) - Real Madrid academy product excelling in Serie A

### Model Confidence

The model is conservative - only 1 player exceeds 60% confidence (Endrick). This reflects:
1. High variance in youth development
2. Many factors beyond skill metrics (injuries, transfers, mental strength)
3. The model prioritizes avoiding false positives

### Recommendations

- **High priority targets:** Endrick (if available), Ethan Nwaneri, Ibrahim Mbaye
- **Monitor closely:** Désiré Doué, Nicolás Paz, George Ilenikhena
- **Development watch:** Players in 50-55% range need another season to confirm trajectory